## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.normal([28 * 28, 256], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([256]))
        self.W2 = tf.Variable(tf.random.normal([256, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))

    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x = tf.reshape(x, [-1, 28 * 28])
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits

model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [10]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 0.49747545 ; accuracy 0.8684667
epoch 1 : loss 0.49716273 ; accuracy 0.8685667
epoch 2 : loss 0.49685088 ; accuracy 0.86863333
epoch 3 : loss 0.49653983 ; accuracy 0.86875
epoch 4 : loss 0.49622965 ; accuracy 0.86875
epoch 5 : loss 0.49592027 ; accuracy 0.8688667
epoch 6 : loss 0.49561185 ; accuracy 0.86895
epoch 7 : loss 0.49530414 ; accuracy 0.86906666
epoch 8 : loss 0.49499726 ; accuracy 0.86911666
epoch 9 : loss 0.49469122 ; accuracy 0.86915
epoch 10 : loss 0.49438602 ; accuracy 0.8692667
epoch 11 : loss 0.4940816 ; accuracy 0.8693
epoch 12 : loss 0.49377796 ; accuracy 0.8693333
epoch 13 : loss 0.49347514 ; accuracy 0.86938334
epoch 14 : loss 0.49317315 ; accuracy 0.86945
epoch 15 : loss 0.49287194 ; accuracy 0.86946666
epoch 16 : loss 0.49257156 ; accuracy 0.8695167
epoch 17 : loss 0.4922719 ; accuracy 0.8695667
epoch 18 : loss 0.49197307 ; accuracy 0.8695833
epoch 19 : loss 0.49167508 ; accuracy 0.8696
epoch 20 : loss 0.49137777 ; accuracy 0.8697
epoch 21 : loss 0.